In [ ]:
import io
import pandas as pd
import ast
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

import xgboost as xgb
import math

In [ ]:
data = pd.read_csv(io.StringIO('''
Width,Length,Slide_Bore,Slide_Stroke,Year,Bore,Stroke
0.9,0.85,100,100,1932,183,181.8
0.9,0.85,0,0,1932,42.39,42.12
0.85,1.2,100,100,1932,214.36,212.97
0.85,1.2,0,0,1932,49.66,49.34
1.25,1,100,100,1932,235.27,233.74
1.25,1,0,0,1932,54.51,54.15
1.8,1.2,100,100,1932,313.7,311.66
1.8,1.2,0,0,1932,72.67,72.2
1.3,0.85,100,100,1932,149.88,223.36
1.3,0.85,0,0,1932,34.72,51.74
1.2,0.95,100,100,1932,112.41,223.36
1.2,0.95,0,0,1932,26.04,51.74
1.2,0.3,100,100,1932,156.85,155.83
1.2,0.3,0,0,1932,36.34,36.1
0.65,0.65,100,100,1932,135.93,135.05
0.65,0.65,0,0,1932,31.49,31.29
0.9,0.85,100,100,1912,195.43,194.93
0.9,0.85,0,0,1912,45.27,45.16
0.85,1.2,100,100,1912,228.93,228.34
0.85,1.2,0,0,1912,53.04,52.9
1.25,1,100,100,1912,251.26,250.62
1.25,1,0,0,1912,58.21,58.06
0.65,0.65,100,100,1912,145.17,144.8
0.65,0.65,0,0,1912,33.63,33.55
1.2,0.3,100,100,1912,167.51,167.08
1.2,0.3,0,0,1912,38.81,38.71
0.9,0.85,100,100,1900,203.29,200.07
0.9,0.85,0,0,1900,47.1,47.09
0.85,1.2,100,100,1900,238.14,234.37
0.85,1.2,0,0,1900,55.17,55.16
1.25,1,100,100,1900,261.37,261.32
1.25,1,0,0,1900,60.55,60.54
0.65,0.65,100,100,1900,151.01,150.98
0.65,0.65,0,0,1900,34.98,34.98
1.2,0.3,100,100,1900,174.25,174.21
1.2,0.3,0,0,1900,40.37,40.36
0.9,0.85,100,100,1958,167.99,166.04
0.9,0.85,0,0,1958,38.92,38.47
0.85,1.2,100,100,1958,196.79,194.51
0.85,1.2,0,0,1958,45.59,45.06
1.25,1,100,100,1958,215.99,213.48
1.25,1,0,0,1958,50.04,49.46
1.8,1.2,100,100,1958,287.98,284.64
1.8,1.2,0,0,1958,66.72,65.94
1.3,0.85,100,100,1958,137.59,204
1.3,0.85,0,0,1958,31.88,47.26
1.2,0.95,100,100,1958,103.19,204
1.2,0.95,0,0,1958,23.91,47.26
1.2,0.3,100,100,1958,143.99,142.32
1.2,0.3,0,0,1958,33.36,32.97
0.65,0.65,100,100,1958,124.79,123.35
0.65,0.65,0,0,1958,28.91,28.58
0.9,0.85,100,100,2000,146.32,143.43
0.9,0.85,0,0,2000,33.9,33.23
0.85,1.2,100,100,2000,171.4,168.01
0.85,1.2,0,0,2000,39.71,38.92
1.25,1,100,100,2000,188.12,184.4
1.25,1,0,0,2000,43.58,42.72
1.8,1.2,100,100,2000,250.83,245.87
1.8,1.2,0,0,2000,58.11,56.96
1.3,0.85,100,100,2000,119.84,176.21
1.3,0.85,0,0,2000,27.76,40.82
1.2,0.95,100,100,2000,89.88,176.21
1.2,0.95,0,0,2000,20.82,40.82
1.2,0.3,100,100,2000,125.41,122.94
1.2,0.3,0,0,2000,29.05,28.48
0.65,0.65,100,100,2000,108.69,106.54
0.65,0.65,0,0,2000,25.18,24.68
'''), header=0)


In [ ]:
data_unique = data[['Width','Length','Year']].drop_duplicates()
for index,row in data_unique.iterrows():
  bore_min = data[((data['Width']==row['Width']) & (data['Length']==row['Length'])) & ((data['Year']==row['Year']) & (data['Slide_Bore']==0))]['Bore'].item()
  bore_max = data[((data['Width']==row['Width']) & (data['Length']==row['Length'])) & (data['Year']==row['Year']) & (data['Slide_Bore']==100)]['Bore'].item()
  stroke_min = data[((data['Width']==row['Width']) & (data['Length']==row['Length'])) & ((data['Year']==row['Year']) & (data['Slide_Stroke']==0))]['Stroke'].item()
  stroke_max = data[((data['Width']==row['Width']) & (data['Length']==row['Length'])) & ((data['Year']==row['Year']) & (data['Slide_Stroke']==100))]['Stroke'].item()
  for i in range(1,100,1):
    bore = bore_min + (i/100)*(bore_max-bore_min)
    stroke = stroke_min + (i/100)*(stroke_max-stroke_min)
    data.loc[len(data)] = [row['Width'],row['Length'],i,i,row['Year'],bore,stroke]

In [ ]:
data['ex_1d0024p_year99'] = 1.0024 **(data['Year']-1899)
data['ex_1d0035p_year99'] = 1.0035 **(data['Year']-1899)
data['ex_1d005p_year99'] = 1.005 **(data['Year']-1899)
data['ex_1d006p_year99'] = 1.006 **(data['Year']-1899)
data['ex_1d008p_year99'] = 1.008 **(data['Year']-1899)
data['ex_1d025p_year99'] = 1.025 **(data['Year']-1899)
data['ex_1d033p_year99'] = 1.033 **(data['Year']-1899)
data['ex_1d038p_year99'] = 1.038 **(data['Year']-1899)
data['ex_1d04p_year99'] = 1.04 **(data['Year']-1899)
data['ex_1d0023p_year99'] = 1.0023 **(data['Year']-1899)
data['ex_1d003p_year99'] = 1.003 **(data['Year']-1899)
data['ex_1d004p_year99'] = 1.004 **(data['Year']-1899)
data['ex_1d0051p_year99'] = 1.0051 **(data['Year']-1899)
data['ex_1d007p_year99'] = 1.007 **(data['Year']-1899)
data['ex_1d01p_year99'] = 1.01 **(data['Year']-1899)
data['ex_1d03p_year99'] = 1.03 **(data['Year']-1899)
data['ex_1d035p_year99'] = 1.035 **(data['Year']-1899)
data['ex_1d039p_year99'] = 1.039 **(data['Year']-1899)
data['ex_1d05p_year99'] = 1.05 **(data['Year']-1899)
data['ex_1d0105p_year99'] = 1.0105 **(data['Year']-1899)
data['Year'] = data['Year'] - 1899

data['displacement'] = (data['Width']+data['Length'])/2

In [ ]:
X_col = ['Width', 'Length', 'Year', 'Bore',
       'Stroke', 'displacement',
         'ex_1d0024p_year99', 'ex_1d0035p_year99',
       'ex_1d005p_year99', 'ex_1d006p_year99', 'ex_1d008p_year99',
       'ex_1d025p_year99', 'ex_1d033p_year99', 'ex_1d038p_year99',
       'ex_1d04p_year99', 'ex_1d0023p_year99', 'ex_1d003p_year99',
       'ex_1d004p_year99', 'ex_1d0051p_year99', 'ex_1d007p_year99',
       'ex_1d01p_year99', 'ex_1d03p_year99', 'ex_1d035p_year99',
       'ex_1d039p_year99', 'ex_1d05p_year99', 'ex_1d0105p_year99'
       ]
Bore_col = ['Width', 'Length', 'Year', 'Bore',
       'displacement',
            'ex_1d0024p_year99', 'ex_1d0035p_year99',
       'ex_1d005p_year99', 'ex_1d006p_year99', 'ex_1d008p_year99',
       'ex_1d025p_year99', 'ex_1d033p_year99', 'ex_1d038p_year99',
       'ex_1d04p_year99', 'ex_1d0023p_year99', 'ex_1d003p_year99',
       'ex_1d004p_year99', 'ex_1d0051p_year99', 'ex_1d007p_year99',
       'ex_1d01p_year99', 'ex_1d03p_year99', 'ex_1d035p_year99',
       'ex_1d039p_year99', 'ex_1d05p_year99', 'ex_1d0105p_year99'
       ]
Stroke_col = ['Width', 'Length', 'Year',
       'Stroke', 'displacement',
              'ex_1d0024p_year99', 'ex_1d0035p_year99',
       'ex_1d005p_year99', 'ex_1d006p_year99', 'ex_1d008p_year99',
       'ex_1d025p_year99', 'ex_1d033p_year99', 'ex_1d038p_year99',
       'ex_1d04p_year99', 'ex_1d0023p_year99', 'ex_1d003p_year99',
       'ex_1d004p_year99', 'ex_1d0051p_year99', 'ex_1d007p_year99',
       'ex_1d01p_year99', 'ex_1d03p_year99', 'ex_1d035p_year99',
       'ex_1d039p_year99', 'ex_1d05p_year99', 'ex_1d0105p_year99'
       ]

In [ ]:
# Define the feature set and target variables
X = data[X_col]
y_bore = data['Slide_Bore']
y_stroke = data['Slide_Stroke']

# Split the data into training and testing sets
X_train, X_test, y_bore_train, y_bore_test, y_stroke_train, y_stroke_test = train_test_split(X, y_bore, y_stroke, test_size=0.2, random_state=42)

# Initialize the models
bore_xgb = xgb.XGBRegressor(objective='reg:squarederror', random_state=42)
stroke_xgb = xgb.XGBRegressor(objective='reg:squarederror', random_state=42)

# Train the models
bore_xgb.fit(X_train[Bore_col], y_bore_train)
stroke_xgb.fit(X_train[Stroke_col], y_stroke_train)

# Make predictions
y_bore_pred_gb = bore_xgb.predict(X_test[Bore_col])
y_stroke_pred_gb = stroke_xgb.predict(X_test[Stroke_col])

# Evaluate the models
bore_rmse_gb = mean_squared_error(y_bore_test, y_bore_pred_gb, squared=False)
bore_r2_gb = r2_score(y_bore_test, y_bore_pred_gb)

stroke_rmse_gb = mean_squared_error(y_stroke_test, y_stroke_pred_gb, squared=False)
stroke_r2_gb = r2_score(y_stroke_test, y_stroke_pred_gb)

print("Bore Model (GB) - RMSE:", bore_rmse_gb)
print("Bore Model (GB) - R2:", bore_r2_gb)

print("Stroke Model (GB) - RMSE:", stroke_rmse_gb)
print("Stroke Model (GB) - R2:", stroke_r2_gb)

Bore Model (GB) - RMSE: 1.2914213509783015
Bore Model (GB) - R2: 0.9980659352834084
Stroke Model (GB) - RMSE: 1.1007467261383743
Stroke Model (GB) - R2: 0.9985948914306525


In [ ]:
test_data = {
    'Width': [0.85],
    'Length': [1.20],
    'Year': [1991],
    'Bore': [122],
    'Stroke': [70],
    'displacement': [2.05/2]
}
test_df = pd.DataFrame(test_data)
test_df['ex_1d0024p_year99'] = 1.0024 **(test_df['Year']-1899)
test_df['ex_1d0035p_year99'] = 1.0035 **(test_df['Year']-1899)
test_df['ex_1d005p_year99'] = 1.005 **(test_df['Year']-1899)
test_df['ex_1d006p_year99'] = 1.006 **(test_df['Year']-1899)
test_df['ex_1d008p_year99'] = 1.008 **(test_df['Year']-1899)
test_df['ex_1d025p_year99'] = 1.025 **(test_df['Year']-1899)
test_df['ex_1d033p_year99'] = 1.033 **(test_df['Year']-1899)
test_df['ex_1d038p_year99'] = 1.038 **(test_df['Year']-1899)
test_df['ex_1d04p_year99'] = 1.04 **(test_df['Year']-1899)
test_df['ex_1d0023p_year99'] = 1.0023 **(test_df['Year']-1899)
test_df['ex_1d003p_year99'] = 1.003 **(test_df['Year']-1899)
test_df['ex_1d004p_year99'] = 1.004 **(test_df['Year']-1899)
test_df['ex_1d0051p_year99'] = 1.0051 **(test_df['Year']-1899)
test_df['ex_1d007p_year99'] = 1.007 **(test_df['Year']-1899)
test_df['ex_1d01p_year99'] = 1.01 **(test_df['Year']-1899)
test_df['ex_1d03p_year99'] = 1.03 **(test_df['Year']-1899)
test_df['ex_1d035p_year99'] = 1.035 **(test_df['Year']-1899)
test_df['ex_1d039p_year99'] = 1.039 **(test_df['Year']-1899)
test_df['ex_1d05p_year99'] = 1.05 **(test_df['Year']-1899)
test_df['ex_1d0105p_year99'] = 1.0105 **(test_df['Year']-1899)
test_df['Year'] = test_df['Year'] - 1899

stroke_xgb.predict(test_df[Stroke_col])

array([18.185667], dtype=float32)

In [ ]:
layouts = pd.read_csv(io.StringIO('''
Name,Year,Design Costs,Costs,Design Requirements,Manufacturing Requirements,Reliability,Power,Fuel,Smooth,Finish,Weight,Width,Length,Skill,Cylinders,Fuel Types,Inductions,Valve,Cylinder Arrangement,Turbine (Circular)
Electric,1892,2.5,3,3,1.5,1.5,0.4,0.6,2,3,20,0.85,1.05,10,"[""Electric""]","[ ""Electric I"", ""Electric II"", ""Electric III"", ""Electric IV"", ""Electric V"" ]",[],1,0,0
Flat,1893,0.55,0.43,0.35,0.25,0.95,0.7,0.45,1.3,0.5,0.55,1.25,1,15,"[""2"",""4"",""6"", ""8"",""10"",""12""]","[ ""Two Stroke"", ""Gasoline"", ""Diesel"", ""Natural Gas"", ""Hybrid"", ""Hydrogen"", ""Autogas"", ""E85"" ]","[ ""Naturally Aspirated"", ""Supercharger"", ""Turbocharger Stage I (Fuel Focused)"", ""Turbocharger Stage I (Power Focused)"", ""Turbocharger Stage II (Fuel Focused)"", ""Turbocharger Stage II (Power Focused)"", ""Turbocharger Stage III (Fuel Focused)"", ""Turbocharger Stage III (Power Focused)"", ""Turbocharger Stage IV (Fuel Focused)"", ""Turbocharger Stage IV (Power Focused)"", ""Twincharger"", ""Hybrid Turbocharger"", ""Variable Geometry Turbocharger"", ""Twin-Turbocharger"", ""Quad-Turbocharger"" ]",2,0,0
H,1895,2.5,2.5,0.65,0.65,0.3,0.15,0.1,1,1,1.6,1.5,1.1,30,"[""4"", ""8"", ""12"", ""16""]","[ ""Two Stroke"", ""Gasoline"", ""Diesel"", ""Natural Gas"", ""Hybrid"", ""Hydrogen"", ""Autogas"", ""E85"" ]","[ ""Naturally Aspirated"", ""Supercharger"", ""Turbocharger Stage I (Fuel Focused)"", ""Turbocharger Stage I (Power Focused)"", ""Turbocharger Stage II (Fuel Focused)"", ""Turbocharger Stage II (Power Focused)"", ""Turbocharger Stage III (Fuel Focused)"", ""Turbocharger Stage III (Power Focused)"", ""Turbocharger Stage IV (Fuel Focused)"", ""Turbocharger Stage IV (Power Focused)"", ""Twincharger"", ""Hybrid Turbocharger"", ""Variable Geometry Turbocharger"", ""Twin-Turbocharger"", ""Quad-Turbocharger"" ]",2,4,0
I,1891,0.3,0.25,0.2,0.15,1.2,0.65,0.7,1,0.3,0.6,0.85,1.2,10,"[""2"",""3"",""4"", ""5"",""6"",""8""]","[ ""Two Stroke"", ""Gasoline"", ""Diesel"", ""Natural Gas"", ""Hybrid"", ""Hydrogen"", ""Autogas"", ""E85"" ]","[ ""Naturally Aspirated"", ""Supercharger"", ""Turbocharger Stage I (Fuel Focused)"", ""Turbocharger Stage I (Power Focused)"", ""Turbocharger Stage II (Fuel Focused)"", ""Turbocharger Stage II (Power Focused)"", ""Turbocharger Stage III (Fuel Focused)"", ""Turbocharger Stage III (Power Focused)"", ""Turbocharger Stage IV (Fuel Focused)"", ""Turbocharger Stage IV (Power Focused)"", ""Twincharger"", ""Hybrid Turbocharger"", ""Variable Geometry Turbocharger"", ""Twin-Turbocharger"", ""Quad-Turbocharger"" ]",2,1,0
Radial,1898,0.7,0.5,0.45,0.45,1.1,0.8,0.3,1,1.1,0.5,1.2,0.3,10,"[""3"",""5"",""7"", ""9"",""15""]","[ ""Two Stroke"", ""Gasoline"", ""Diesel"", ""Natural Gas"", ""Hybrid"", ""Hydrogen"", ""Autogas"", ""E85"" ]","[ ""Naturally Aspirated"", ""Supercharger"", ""Turbocharger Stage I (Fuel Focused)"", ""Turbocharger Stage I (Power Focused)"", ""Turbocharger Stage II (Fuel Focused)"", ""Turbocharger Stage II (Power Focused)"", ""Turbocharger Stage III (Fuel Focused)"", ""Turbocharger Stage III (Power Focused)"", ""Turbocharger Stage IV (Fuel Focused)"", ""Turbocharger Stage IV (Power Focused)"", ""Twincharger"", ""Hybrid Turbocharger"", ""Variable Geometry Turbocharger"", ""Twin-Turbocharger"", ""Quad-Turbocharger"" ]",3,-1,0
Rotary,1895,0.7,0.4,0.4,0.4,1.2,0.8,0.2,1.1,1,0.5,1.2,0.3,10,"[""3"",""5"",""7"", ""9"",""15""]","[ ""Two Stroke"", ""Gasoline"", ""Diesel"", ""Natural Gas"", ""Hybrid"", ""Hydrogen"", ""Autogas"", ""E85"" ]","[ ""Naturally Aspirated"", ""Supercharger"", ""Turbocharger Stage I (Fuel Focused)"", ""Turbocharger Stage I (Power Focused)"", ""Turbocharger Stage II (Fuel Focused)"", ""Turbocharger Stage II (Power Focused)"", ""Turbocharger Stage III (Fuel Focused)"", ""Turbocharger Stage III (Power Focused)"", ""Turbocharger Stage IV (Fuel Focused)"", ""Turbocharger Stage IV (Power Focused)"", ""Twincharger"", ""Hybrid Turbocharger"", ""Variable Geometry Turbocharger"", ""Twin-Turbocharger"", ""Quad-Turbocharger"" ]",3,-1,0
Single,1890,0.25,0.1,0.15,0.15,1,0.25,0.8,-1,0.15,0.1,0.65,0.65,0,"[""Cylinder""]","[ ""Two Stroke"", ""Gasoline"", ""Diesel"", ""Natural Gas"", ""Hybrid"", ""Hydrogen"", ""Autogas"", ""E85"" ]","[ ""Naturally Aspirated"", ""Supercharger"", ""Turbocharger Stage I (Fuel Focused)"", ""Turbocharger Stage I (Power Focused)"", ""Turbocharger Stage II (Fuel Focused)"", ""Turbocharger Stage II (Power Focused)"", ""Turbocharger Stage III (Fuel Focused)"", ""Turbocharger Stage III (Power Focused)"", ""Turbocharger Stage IV (Fuel Focused)"", ""Turbocharger Stage IV (Power Focused)"", ""Twincharger"", ""Hybrid Turbocharger"", ""Variable Geometry Turbocharger"", ""Twin-Turbocharger"", ""Quad-Turbocharger"" ]",2,0,0
Steam,1870,0.22,0.1,0.1,0.15,0.6,0.8,0.05,2,1,2.5,1.5,1.5,10,"[""Steam""]","[""Water""]","[ ""Naturally Aspirated"", ""Supercharger"", ""Turbocharger Stage I (Fuel Focused)"", ""Turbocharger Stage I (Power Focused)"", ""Turbocharger Stage II (Fuel Focused)"", ""Turbocharger Stage II (Power Focused)"", ""Turbocharger Stage III (Fuel Focused)"", ""Turbocharger Stage III (Power Focused)"", ""Turbocharger Stage IV (Fuel Focused)"", ""Turbocharger Stage IV (Power Focused)"", ""Twincharger"", ""Hybrid Turbocharger"", ""Variable Geometry Turbocharger"", ""Twin-Turbocharger"", ""Quad-Turbocharger"" ]",1,0,0
U,1894,0.75,0.65,0.4,0.4,0.4,0.6,0.35,0.75,0.6,1.25,1.8,1.2,50,"[""4"",""6"", ""8"",""10"",""12""]","[ ""Two Stroke"", ""Gasoline"", ""Diesel"", ""Natural Gas"", ""Hybrid"", ""Hydrogen"", ""Autogas"", ""E85"" ]","[ ""Naturally Aspirated"", ""Supercharger"", ""Turbocharger Stage I (Fuel Focused)"", ""Turbocharger Stage I (Power Focused)"", ""Turbocharger Stage II (Fuel Focused)"", ""Turbocharger Stage II (Power Focused)"", ""Turbocharger Stage III (Fuel Focused)"", ""Turbocharger Stage III (Power Focused)"", ""Turbocharger Stage IV (Fuel Focused)"", ""Turbocharger Stage IV (Power Focused)"", ""Twincharger"", ""Hybrid Turbocharger"", ""Variable Geometry Turbocharger"", ""Twin-Turbocharger"", ""Quad-Turbocharger"" ]",2,0,0
V,1892,0.4,0.35,0.3,0.35,1,0.75,0.5,0.7,0.4,0.6,0.9,0.85,15,"[""2"",""4"",""6"", ""8"",""10"",""12"",""16""]","[ ""Two Stroke"", ""Gasoline"", ""Diesel"", ""Natural Gas"", ""Hybrid"", ""Hydrogen"", ""Autogas"", ""E85"" ]","[ ""Naturally Aspirated"", ""Supercharger"", ""Turbocharger Stage I (Fuel Focused)"", ""Turbocharger Stage I (Power Focused)"", ""Turbocharger Stage II (Fuel Focused)"", ""Turbocharger Stage II (Power Focused)"", ""Turbocharger Stage III (Fuel Focused)"", ""Turbocharger Stage III (Power Focused)"", ""Turbocharger Stage IV (Fuel Focused)"", ""Turbocharger Stage IV (Power Focused)"", ""Twincharger"", ""Hybrid Turbocharger"", ""Variable Geometry Turbocharger"", ""Twin-Turbocharger"", ""Quad-Turbocharger"" ]",2,0,0
VV,1995,2.3,2.9,0.7,0.7,0.55,1,0.2,0.45,1,1.5,1.85,0.85,70,"[""8"", ""12"", ""16""]","[ ""Two Stroke"", ""Gasoline"", ""Diesel"", ""Natural Gas"", ""Hybrid"", ""Hydrogen"", ""Autogas"", ""E85"" ]","[ ""Naturally Aspirated"", ""Supercharger"", ""Turbocharger Stage I (Fuel Focused)"", ""Turbocharger Stage I (Power Focused)"", ""Turbocharger Stage II (Fuel Focused)"", ""Turbocharger Stage II (Power Focused)"", ""Turbocharger Stage III (Fuel Focused)"", ""Turbocharger Stage III (Power Focused)"", ""Turbocharger Stage IV (Fuel Focused)"", ""Turbocharger Stage IV (Power Focused)"", ""Twincharger"", ""Hybrid Turbocharger"", ""Variable Geometry Turbocharger"", ""Twin-Turbocharger"", ""Quad-Turbocharger"" ]",2,4,0
W,1903,2.1,2,1,0.75,0.9,1,0.35,0.9,1,1.5,1.3,0.85,40,"[""3"",""6"",""9"", ""12"",""18""]","[ ""Two Stroke"", ""Gasoline"", ""Diesel"", ""Natural Gas"", ""Hybrid"", ""Hydrogen"", ""Autogas"", ""E85"" ]","[ ""Naturally Aspirated"", ""Supercharger"", ""Turbocharger Stage I (Fuel Focused)"", ""Turbocharger Stage I (Power Focused)"", ""Turbocharger Stage II (Fuel Focused)"", ""Turbocharger Stage II (Power Focused)"", ""Turbocharger Stage III (Fuel Focused)"", ""Turbocharger Stage III (Power Focused)"", ""Turbocharger Stage IV (Fuel Focused)"", ""Turbocharger Stage IV (Power Focused)"", ""Twincharger"", ""Hybrid Turbocharger"", ""Variable Geometry Turbocharger"", ""Twin-Turbocharger"", ""Quad-Turbocharger"" ]",2,3,0
Wankel,1947,1,0.5,0.5,0.5,0.75,0.35,0.45,1.2,0.45,0.35,0.7,0.6,60,"[""Wankel""]","[ ""Gasoline"", ""Diesel"", ""Natural Gas"", ""Hybrid"", ""Hydrogen"", ""Autogas"" ]","[ ""Naturally Aspirated"", ""Supercharger"", ""Turbocharger Stage I (Fuel Focused)"", ""Turbocharger Stage I (Power Focused)"", ""Turbocharger Stage II (Fuel Focused)"", ""Turbocharger Stage II (Power Focused)"", ""Turbocharger Stage III (Fuel Focused)"", ""Turbocharger Stage III (Power Focused)"", ""Turbocharger Stage IV (Fuel Focused)"", ""Turbocharger Stage IV (Power Focused)"", ""Twincharger"", ""Hybrid Turbocharger"", ""Variable Geometry Turbocharger"", ""Twin-Turbocharger"", ""Quad-Turbocharger"" ]",1,0,1
X,1905,2.4,2,0.8,0.8,0.25,0.8,0.15,0.9,1,1.6,1.2,0.95,35,"[""4"", ""8"", ""12"", ""16""]","[ ""Two Stroke"", ""Gasoline"", ""Diesel"", ""Natural Gas"", ""Hybrid"", ""Hydrogen"", ""Autogas"", ""E85"" ]","[ ""Naturally Aspirated"", ""Supercharger"", ""Turbocharger Stage I (Fuel Focused)"", ""Turbocharger Stage I (Power Focused)"", ""Turbocharger Stage II (Fuel Focused)"", ""Turbocharger Stage II (Power Focused)"", ""Turbocharger Stage III (Fuel Focused)"", ""Turbocharger Stage III (Power Focused)"", ""Turbocharger Stage IV (Fuel Focused)"", ""Turbocharger Stage IV (Power Focused)"", ""Twincharger"", ""Hybrid Turbocharger"", ""Variable Geometry Turbocharger"", ""Twin-Turbocharger"", ""Quad-Turbocharger"" ]",2,4,0
'''), header=0)

cylinders = pd.read_csv(io.StringIO('''
Name,Year,Cost,Design Costs,Design Req.,Manu. Req.,Finish,Skill,Power,Fuel,Reliability,Weight,Number of Cylinders,Smoothness
Steam,1890,0.2,0.2,0.2,0.2,1,0,0.1,0.3,1,1.8,2,1.2
Electric,1890,2,2,3,1.5,3,0,1,0.75,1.5,20,4,1.5
Wankle,1890,0.8,0.5,0.75,0.75,1.2,0,0.6,0.7,0.8,0.3,4,1
Cylinder,1890,0.1,0.1,0.1,0.1,0.1,0,0.1,1,1,0.1,1,-1
2,1890,0.15,0.15,0.15,0.15,0.12,0,0.15,1.2,1.03,0.2,2,0
3,1890,0.2,0.2,0.2,0.2,0.15,0,0.25,1.1,1.04,0.3,3,0.22
4,1890,0.25,0.25,0.25,0.25,0.18,0,0.4,1,1.05,0.4,4,0.5
5,1890,0.3,0.3,0.3,0.3,0.22,10,0.55,0.95,1.03,0.5,5,0.7
6,1890,0.35,0.35,0.35,0.35,0.27,10,0.75,0.85,1,0.6,6,0.85
7,1890,0.4,0.4,0.4,0.4,0.33,20,0.85,0.8,1,0.7,7,0.9
8,1890,0.45,0.45,0.45,0.5,0.4,20,1,0.75,1,0.8,8,1
9,1890,0.5,0.5,0.5,0.5,0.48,30,1.25,0.7,0.8,0.9,9,0.8
10,1900,0.6,0.6,0.6,1.4,0.57,35,1.5,0.7,0.65,1,10,1.2
12,1902,0.7,0.7,0.7,1.2,0.67,40,1.6,0.6,0.5,1.2,12,1.8
15,1904,0.8,0.8,0.8,0.8,0.78,50,1.75,0.5,0.45,1.5,15,0.8
16,1906,0.9,0.9,0.9,1.9,0.9,55,1.8,0.4,0.4,1.6,16,1
18,1920,1,1,1,1,1,70,2,0.3,0.3,1.8,18,0.8
'''), header=0)

Fuel = pd.read_csv(io.StringIO('''
Name,Year,Cost,Design Costs,Design Req.,Manu. Req.,Skill,Finish,Power,Fuel,Reliability,Weight,RPM,Smoothness
Autogas,1960,1.2,1.4,1.4,1.4,40,1,0.55,0.4,0.9,1,1.1,0.8
Diesel,1912,1.1,1.1,0.8,0.55,25,1,5,2.5,2,1.2,0.5,0.7
E85,1900,1.25,1.5,1.5,1.25,1,1.1,0.7,0.25,0.8,1.1,1.2,0.9
Electric I,1890,1,1,5,2.5,0,2.5,0.1,0.01,0.01,30,0.25,1.5
Electric II,1930,1.5,1.2,3,2.5,20,3,0.2,0.02,0.02,25,0.35,1.5
Electric III,1970,3,1.5,3,2.5,30,3,0.3,0.4,0.1,20,0.4,1.5
Electric IV,1990,4.5,2,3,2,40,3,0.6,20,0.3,25,0.5,1.5
Electric V,2005,12,4,2.5,2,60,3,2.5,80,2.5,25,1.5,1.5
Gasoline,1890,1,0.8,0.8,0.75,0,1,0.6,1,1.2,1,1.2,1
Hybrid,1985,4,5,3,2,60,1.2,0.55,3.4,1,5,1,1.3
Hydrogen,1980,5,5,5,3,70,1.1,0.25,1.5,0.6,1,1,0.75
Natural Gas,1960,2.6,2.6,2.6,1.5,50,2,0.3,3.5,0.7,1,0.8,1
Water,1890,1,1,1,1,0,1,2,0.01,0.5,8,0.4,1
'''), header=0)

Valvetrain = pd.read_csv(io.StringIO('''
Name,Year,Valve Type,Costs,Design Costs,Design Req.,Manu. Req.,Skill,Finish,Fuel,Reliability,Weight,RPM,Smoothness,Size,Power
No Valve,1890,1,0.1,1,1,1,0,1,1,1,0.3,1,1.4,1,1
DOHC,1905,2,1.2,1.2,1.3,1.7,50,1.2,1.1,1.2,0.9,1.08,1.2,1.2,1
F Head,1890,2,0.6,0.7,0.9,0.9,20,1,0.75,1,0.75,0.85,0.85,0.7,1
L Head,1890,2,0.4,0.5,0.6,0.5,0,0.7,0.5,0.6,0.7,0.6,0.7,1.05,1
OHV,1900,2,0.6,0.6,0.9,0.9,30,1,0.85,1.1,0.7,0.95,0.85,0.8,1
Poppet Valve,1890,3,0.7,0.6,0.7,0.75,0,1,0.4,1,1,0.45,1,1,1
SOHC,1890,2,1,1.1,1.2,1.5,35,1.1,1,1.1,0.7,1.05,1.1,1.1,1
Sleeve Valve,1890,3,1,0.8,0.9,1,25,1.2,0.55,0.75,0.8,0.55,1.3,0.9,1
T Head,1890,2,0.5,0.6,0.7,0.8,15,0.8,0.5,0.8,0.7,0.67,0.85,1.07,1
Two Stroke,1890,2,0.3,0.3,0.3,0.3,0,0.6,0.35,-1,-1,1,-1,0,0.5
'''), header=0)

Induction = pd.read_csv(io.StringIO('''
Name,Year,Cost,Design Costs,Design Req.,Manu. Req.,Skill,Finish,Power,Fuel,Reliability,Weight
Hybrid Turbocharger,1985,5,3,2,1.4,50,1.1,2.5,1.05,0.65,1.2
Naturally Aspirated,1890,0,1,1,1,0,1,1,1,1,1
No Induction,1890,0,1,1,1,0,1,1,1,1,1
Quad-Turbocharger,1972,6,2.5,2.1,3.1,70,1.7,2.4,1.2,0.75,1.1
Supercharger,1902,0.4,1.05,1.05,1.05,20,1,1.3,1.02,1,1.05
Turbocharger Stage I (Fuel Focused),1932,0.5,1.07,1.05,1.05,20,1.05,1.1,1.25,0.9,1.05
Turbocharger Stage I (Power Focused),1932,0.5,1.07,1.05,1.05,20,1.05,1.3,1.1,0.9,1.05
Turbocharger Stage II (Fuel Focused),1952,0.65,1.1,1.1,1.1,27,1.05,1.2,1.45,0.9,1.05
Turbocharger Stage II (Power Focused),1952,0.65,1.1,1.1,1.1,27,1.05,1.6,1.2,0.9,1.05
Turbocharger Stage III (Fuel Focused),1972,0.8,1.15,1.15,1.15,32,1.05,1.3,1.7,0.87,1.05
Turbocharger Stage III (Power Focused),1972,0.8,1.15,1.15,1.15,32,1.05,1.9,1.3,0.87,1.05
Turbocharger Stage IV (Fuel Focused),1992,0.85,1.2,1.2,1.2,40,1.05,1.4,1.85,0.85,1.05
Turbocharger Stage IV (Power Focused),1992,1,1.2,1.2,1.2,40,1.05,2.2,1.45,0.85,1.05
Twin-Turbocharger,1962,2.5,1.75,1.5,2.15,45,1.3,2.1,1.25,0.75,1.1
Twincharger,1975,1.1,1.35,1.3,1.32,20,1.15,1.8,1.25,0.9,1.1
Variable Geometry Turbocharger,1932,1.45,1.35,1.3,1.2,25,1.1,1.4,1.05,1,1.3
'''), header=0)

world_event = pd.read_csv(io.StringIO('''
year,interest_rate,carprice_rate
1900,1.02,1.4
1901,1.02626,1.4129
1902,1.03108,1.42581
1903,1.02008,1.43871
1904,1.01077,1.45161
1905,1.02,1.46452
1906,1.02923,1.47742
1907,1.03846,1.49032
1908,1.01129,1.38492
1909,1.01479,1.32459
1910,1.01783,1.26426
1911,1.01633,1.20393
1912,1.01997,1.14361
1913,1.01964,1.08328
1914,1.01581,1.02295
1915,1.01063,0.962623
1916,1.01823,0.902295
1917,1.02583,0.841967
1918,1.03343,0.781639
1919,1.04103,0.721311
1920,1.04717,0.660984
1921,1.03717,0.600656
1922,1.02833,0.540328
1923,1.031,0.48
1924,1.0204,0.464706
1925,1.02516,0.4
1926,1.02992,0.40625
1927,1.03468,0.4125
1928,1.03944,0.41875
1929,1.04421,0.425
1930,1.0181,0.43125
1931,1.01238,0.4375
1932,1.01041,0.44375
1933,1.0111,0.45
1934,1.0118,0.45
1935,1.01249,0.45
1936,1.01319,0.45
1937,1.01388,0.45
1938,1.01458,0.45
1939,1.01528,0.45
1940,1.01597,0.45
1941,1.01667,0.451351
1942,1.01736,0.452703
1943,1.01806,0.454054
1944,1.01875,0.455405
1945,1.01945,0.456757
1946,1.02014,0.458108
1947,1.02084,0.459459
1948,1.02154,0.460811
1949,1.02223,0.462162
1950,1.02293,0.463514
1951,1.02362,0.464865
1952,1.02432,0.466216
1953,1.02501,0.467568
1954,1.02571,0.468919
1955,1.02641,0.47027
1956,1.0271,0.471622
1957,1.0278,0.472973
1958,1.02849,0.474324
1959,1.02919,0.475676
1960,1.02988,0.477027
1961,1.03559,0.478378
1962,1.0423,0.47973
1963,1.04901,0.481081
1964,1.05572,0.482432
1965,1.06243,0.483784
1966,1.06914,0.485135
1967,1.07585,0.486486
1968,1.08117,0.487838
1969,1.07088,0.489189
1970,1.05737,0.490541
1971,1.04746,0.491892
1972,1.07696,0.493243
1973,1.10252,0.494595
1974,1.0847,0.495946
1975,1.06688,0.497297
1976,1.06173,0.498649
1977,1.08193,0.5
1978,1.10213,0.553846
1979,1.12233,0.607692
1980,1.14253,0.661538
1981,2.19208,0.715385
1982,1.61708,0.769231
1983,1.091,0.823077
1984,1.10124,0.876923
1985,1.0921,0.930769
1986,1.08295,0.984615
1987,1.07381,1.03846
1988,1.07667,1.09231
1989,1.0881,1.14615
1990,1.07689,1.2
1991,1.06116,1.20334
1992,1.04542,1.20669
1993,1.032,1.21003
1994,1.044,1.21337
1995,1.056,1.21671
1996,1.05527,1.22006
1997,1.052,1.2234
1998,1.04954,1.22674
1999,1.05604,1.23008
2000,1.06056,1.23343
2001,1.04322,1.23677
2002,1.02589,1.24011
2003,1.01111,1.24345
2004,1.02444,1.2468
2005,1.03778,1.25014
2006,1.04923,1.25348
2007,1.04,1.25682
2008,1.03077,1.26017
2009,1.02154,1.26351
2010,1.0258,1.26685
2011,1.03275,1.27019
2012,1.03971,1.27354
2013,1.04667,1.27688
2014,1.05362,1.28022
2015,1.05972,1.28357
2016,1.05639,1.28691
2017,1.05306,1.29025
2018,1.04972,1.29359
2019,1.04639,1.29694
'''), header=0)


In [ ]:
# Function to convert string representation of list back to list
def str_to_list(s):
    return ast.literal_eval(s)

In [ ]:
# Apply the function to the columns
layouts["Cylinders"] = layouts["Cylinders"].apply(str_to_list)
layouts["Fuel Types"] = layouts["Fuel Types"].apply(str_to_list)
layouts["Inductions"] = layouts["Inductions"].apply(str_to_list)

In [ ]:
layouts.head()

,Name,Year,Design Costs,Costs,Design Requirements,Manufacturing Requirements,Reliability,Power,Fuel,Smooth,...,Weight,Width,Length,Skill,Cylinders,Fuel Types,Inductions,Valve,Cylinder Arrangement,Turbine (Circular)
0,Electric,1892,2.50,3.00,3.00,1.50,1.50,0.40,0.60,2.0,...,20.00,0.85,1.05,10,[Electric],"[Electric I, Electric II, Electric III, Electr...",[],1,0,0
1,Flat,1893,0.55,0.43,0.35,0.25,0.95,0.70,0.45,1.3,...,0.55,1.25,1.00,15,"[2, 4, 6, 8, 10, 12]","[Two Stroke, Gasoline, Diesel, Natural Gas, Hy...","[Naturally Aspirated, Supercharger, Turbocharg...",2,0,0
2,H,1895,2.50,2.50,0.65,0.65,0.30,0.15,0.10,1.0,...,1.60,1.50,1.10,30,"[4, 8, 12, 16]","[Two Stroke, Gasoline, Diesel, Natural Gas, Hy...","[Naturally Aspirated, Supercharger, Turbocharg...",2,4,0
3,I,1891,0.30,0.25,0.20,0.15,1.20,0.65,0.70,1.0,...,0.60,0.85,1.20,10,"[2, 3, 4, 5, 6, 8]","[Two Stroke, Gasoline, Diesel, Natural Gas, Hy...","[Naturally Aspirated, Supercharger, Turbocharg...",2,1,0
4,Radial,1898,0.70,0.50,0.45,0.45,1.10,0.80,0.30,1.0,...,0.50,1.20,0.30,10,"[3, 5, 7, 9, 15]","[Two Stroke, Gasoline, Diesel, Natural Gas, Hy...","[Naturally Aspirated, Supercharger, Turbocharg...",3,-1,0


In [ ]:
# Setup
year = 1990
max_cost = 400
Global_Interest_Rate = world_event[world_event['year']==math.floor(year)]['interest_rate'].item()
carPriceRate = world_event[world_event['year']==math.floor(year)]['carprice_rate'].item()

In [ ]:
Slider_Performance_Torque = 0/100
Slider_Performance_FuelEconomy = 0/100
Slider_DesignFocus_FuelEconomy = 0/100
Slider_DesignFocus_Performance = 0/100
Slider_Technology_Components = 0/100
Slider_Technology_Materials = 0/100
Slider_Technology_Technologies = 0/100
Slider_Technology_Techniques = 0/100
Slider_Performance_Revolutions = 0/100
Slider_Layout_Weight = 0/100

Slider_Layout_Bore = 100/100
Slider_Layout_Stroke = 100/100

Slider_Layout_Length = 100/100
Slider_Layout_Width = 100/100
Slider_DesignFocus_Dependability = 0/100

# I 6 T Head Natural
layout_name = 'I'
Cylinders_name = '6'
Fuel_name = 'Gasoline'
Induction_name = 'Naturally Aspirated'
Valvetrain_name = 'T Head'
SubComponent_Layout_Length = layouts[layouts['Name']==layout_name]['Length'].item()
SubComponent_Layout_Width = layouts[layouts['Name']==layout_name]['Width'].item()
SubComponent_Layout_PowerRatings = layouts[layouts['Name']==layout_name]['Power'].item()
SubComponent_Cylinders_PowerRating = cylinders[cylinders['Name']==Cylinders_name]['Power'].item()
SubComponent_Cylinders_CylinderCount = cylinders[cylinders['Name']==Cylinders_name]['Number of Cylinders'].item()
SubComponent_Induction_PowerRating = Induction[Induction['Name']==Induction_name]['Power'].item()
SubComponent_Valve_PowerRating = Valvetrain[Valvetrain['Name']==Valvetrain_name]['Power'].item()
SubComponent_Valve_RPM = Valvetrain[Valvetrain['Name']==Valvetrain_name]['RPM'].item()
SubComponent_FuelType_RPM = Fuel[Fuel['Name']==Fuel_name]['RPM'].item()
SubComponent_FuelType_PowerRating = Fuel[Fuel['Name']==Fuel_name]['Power'].item()

SubComponent_Cylinders_UnitCosts = 0.35
SubComponent_Layout_UnitCosts = 	0.35
SubComponent_Valve_UnitCosts = 	0.5
SubComponent_Induction_UnitCosts = 	0
SubComponent_FuelType_UnitCosts = 	1

In [ ]:
# Calculate year factor
if year > 2020:
  ex_0d996p_year50R = 0.901037361
else:
  ex_0d996p_year50R = 0.996**(2050-year)

AdjustedYear = year - 1899
ex_1d0024p_year99 = 1.0024 **(year-1899)
ex_1d0035p_year99 = 1.0035 **(year-1899)
ex_1d005p_year99 = 1.005 **(year-1899)
ex_1d006p_year99 = 1.006 **(year-1899)
ex_1d008p_year99 = 1.008 **(year-1899)
ex_1d025p_year99 = 1.025 **(year-1899)
ex_1d033p_year99 = 1.033 **(year-1899)
ex_1d038p_year99 = 1.038 **(year-1899)
ex_1d04p_year99 = 1.04 **(year-1899)
ex_1d0023p_year99 = 1.0023 **(year-1899)
ex_1d003p_year99 = 1.003 **(year-1899)
ex_1d004p_year99 = 1.004 **(year-1899)
ex_1d0051p_year99 = 1.0051 **(year-1899)
ex_1d007p_year99 = 1.007 **(year-1899)
ex_1d01p_year99 = 1.01 **(year-1899)
ex_1d03p_year99 = 1.03 **(year-1899)
ex_1d035p_year99 = 1.035 **(year-1899)
ex_1d039p_year99 = 1.039 **(year-1899)
ex_1d05p_year99 = 1.05 **(year-1899)
ex_1d0105p_year99 = 1.0105 **(year-1899)

In [ ]:
# Fix value
designRandomVal = 1
Marq_DesignEngineSkill = 100
Slider_Layout_Displacement = (Slider_Layout_Bore + Slider_Layout_Stroke) / 2.0

In [ ]:
def cal_optimal_performance(Bore_mm,Stroke_mm):
  Displacement_CC = (0.7854 * ((Bore_mm/10) * (Bore_mm/10)) * (Stroke_mm/10) * SubComponent_Cylinders_CylinderCount)
  #Torque
  Torque = 10 + (Marq_DesignEngineSkill/20.0) + \
    ((((25) * ((Slider_Performance_Torque - 0.4)*1.5)*ex_1d01p_year99) + \
    ((4*(SubComponent_Layout_Length + SubComponent_Layout_Width) )*ex_1d005p_year99) - \
    (14 * (Slider_Performance_FuelEconomy+Slider_DesignFocus_FuelEconomy) * ex_1d004p_year99) +\
    (SubComponent_Layout_PowerRatings*5 + SubComponent_Cylinders_PowerRating*13 +\
    SubComponent_FuelType_PowerRating*24 + 100*SubComponent_Induction_PowerRating +\
    (5 * ex_1d004p_year99 * Slider_DesignFocus_Performance) +\
    8*(Slider_Technology_Components+Slider_Technology_Materials + \
    Slider_Technology_Technologies +Slider_Technology_Techniques))*ex_1d0024p_year99))

  Torque = Torque * ((SubComponent_Cylinders_CylinderCount * Stroke_mm*0.93 * Bore_mm*0.9)*0.000027)+5

  if year < 2050:
      Torque = Torque * ex_0d996p_year50R

  Torque = Torque * SubComponent_Valve_PowerRating

  #RPM
  tmpAY = AdjustedYear
  if tmpAY > 80:
      tmpAY = 80 + ((AdjustedYear-80)/5)


  rpm = ((((tmpAY**4)*0.00000420875) - \
      ((19*(tmpAY**3))*0.00016835) + ((427*(tmpAY**2))*0.00126) +\
      ((1315*(tmpAY))*0.01515) + 620 ) + (265 * ex_1d01p_year99 * Slider_DesignFocus_Performance) +\
      (465 * ex_1d0105p_year99 * (Slider_Performance_Revolutions*5.5)) -\
      (10 * ex_1d01p_year99 * SubComponent_Induction_PowerRating) +\
      (55 * ex_1d005p_year99 * (1-Slider_Layout_Weight)) - (30* ex_1d005p_year99 *\
      (Slider_DesignFocus_FuelEconomy + Slider_Performance_FuelEconomy))+  \
      (25 * ex_1d01p_year99 * Slider_Technology_Components) + \
      (25 * ex_1d01p_year99 * Slider_Technology_Materials) + \
      (25 * ex_1d01p_year99 * Slider_Technology_Technologies)) * SubComponent_FuelType_RPM


  rpm = rpm * SubComponent_Valve_RPM

  rpm = rpm - ((rpm/1.5) * (Stroke_mm/221.136364))

  if rpm < 25:
      rpm = 25
  hp = (Torque * rpm) / 5252
  # Unit cost
  Unit_Costs =((((((70* ex_1d01p_year99 * (((1-Slider_Layout_Length) + (1-Slider_Layout_Width))/2.0)) +\
    (220 * ex_1d004p_year99 * (((0.25+(Slider_Performance_Revolutions * \
    Slider_Performance_Revolutions) + (Slider_Performance_Torque * Slider_Performance_Torque))/2.0) -\
    (0.5-(Slider_Performance_FuelEconomy*Slider_Performance_FuelEconomy)) )  ) +\
    (60 * ex_1d01p_year99) *  ((Slider_Performance_Revolutions * Slider_Performance_Revolutions) +\
    (Slider_Performance_Torque * Slider_Performance_Torque)) +\
    220 * ex_1d008p_year99*(0.1+(((Slider_Technology_Materials*Slider_Technology_Materials)+\
    (Slider_Technology_Techniques*Slider_Technology_Techniques ) + \
    (Slider_Technology_Components*Slider_Technology_Components))) ) +\
    170 * ex_1d008p_year99*(Slider_Technology_Technologies*Slider_Technology_Technologies) +\
    50 * ex_1d0035p_year99 * (Slider_DesignFocus_Dependability * Slider_DesignFocus_Dependability) +\
    180 * ex_1d0035p_year99 * (Slider_DesignFocus_Performance * Slider_DesignFocus_Performance )+\
    (260 * ex_1d006p_year99 * (2.168 * Slider_Layout_Displacement**1.5 -4.44 * Slider_Layout_Displacement**3 +\
    2.646  * Slider_Layout_Displacement**4.5 + 3.126 * Slider_Layout_Displacement**6 )+\
    (70 * ex_1d005p_year99 * (SubComponent_Cylinders_CylinderCount/6.0) + \
    (.75 +  Slider_Layout_Displacement**1.5) - (Slider_Layout_Weight**2)) + 10 * \
    (Slider_DesignFocus_FuelEconomy**2) - 50 ) *  ex_1d003p_year99 + \
    (160 * SubComponent_Cylinders_UnitCosts)**ex_1d003p_year99 +\
    (120 * SubComponent_Layout_UnitCosts)**ex_1d004p_year99 +\
    (140 * SubComponent_Valve_UnitCosts)**ex_1d004p_year99 +\
    (435 * SubComponent_Induction_UnitCosts)**ex_1d004p_year99 +\
    (120 * SubComponent_FuelType_UnitCosts)**ex_1d004p_year99) * \
    (.125 + 0.12 * SubComponent_Cylinders_CylinderCount)) * \
    (Global_Interest_Rate/2.0)) + 50)  * carPriceRate) * (designRandomVal)

  Hyper_Sliders = ((Slider_Layout_Displacement*2 + (1-Slider_Layout_Length) + \
      (1-Slider_Layout_Width) + (1-Slider_Layout_Weight)) +\
      (Slider_Performance_Revolutions + Slider_Performance_Torque + Slider_Performance_FuelEconomy) +\
      (Slider_DesignFocus_Performance + Slider_DesignFocus_FuelEconomy  + Slider_DesignFocus_Dependability) +\
      (Slider_Technology_Materials + Slider_Technology_Components + Slider_Technology_Techniques +\
      Slider_Technology_Technologies))/13.0

  Hyper_Costs = 475 * ex_1d04p_year99 * (Hyper_Sliders*Hyper_Sliders*Hyper_Sliders*Hyper_Sliders)


  Unit_Costs = Unit_Costs + Hyper_Costs - ((Unit_Costs/10) * (Marq_DesignEngineSkill/100))

  return Displacement_CC,Torque*1.356,hp,Unit_Costs

In [ ]:
cal_optimal_performance(176,173)

(25253.035315200006, 665.3489293569729, 157.4507036425835, 2033.7153313189683)

In [ ]:
year = 1991
AdjustedYear = year - 1899
ex_1d0024p_year99 = 1.0024 **(year-1899)
ex_1d0035p_year99 = 1.0035 **(year-1899)
ex_1d005p_year99 = 1.005 **(year-1899)
ex_1d006p_year99 = 1.006 **(year-1899)
ex_1d008p_year99 = 1.008 **(year-1899)
ex_1d025p_year99 = 1.025 **(year-1899)
ex_1d033p_year99 = 1.033 **(year-1899)
ex_1d038p_year99 = 1.038 **(year-1899)
ex_1d04p_year99 = 1.04 **(year-1899)
ex_1d0023p_year99 = 1.0023 **(year-1899)
ex_1d003p_year99 = 1.003 **(year-1899)
ex_1d004p_year99 = 1.004 **(year-1899)
ex_1d0051p_year99 = 1.0051 **(year-1899)
ex_1d007p_year99 = 1.007 **(year-1899)
ex_1d01p_year99 = 1.01 **(year-1899)
ex_1d03p_year99 = 1.03 **(year-1899)
ex_1d035p_year99 = 1.035 **(year-1899)
ex_1d039p_year99 = 1.039 **(year-1899)
ex_1d05p_year99 = 1.05 **(year-1899)
ex_1d0105p_year99 = 1.0105 **(year-1899)
if year > 2020:
  ex_0d996p_year50R = 0.901037361
else:
  ex_0d996p_year50R = 0.996**(2050-year)

SubComponent_Cylinders_CylinderCount = 2

Torque = 10 + (Marq_DesignEngineSkill/20.0) + \
    ((((25) * ((Slider_Performance_Torque - 0.4)*1.5)*ex_1d01p_year99) + \
    ((4*(SubComponent_Layout_Length + SubComponent_Layout_Width) )*ex_1d005p_year99) - \
    0 +\
    (SubComponent_Layout_PowerRatings*5 + SubComponent_Cylinders_PowerRating*13 +\
    SubComponent_FuelType_PowerRating*24 + 100*SubComponent_Induction_PowerRating +\
    0 +\
    0)*ex_1d0024p_year99))

Torque = Torque * ((SubComponent_Cylinders_CylinderCount * 173*0.93 * 176*0.9)*0.000027)+5
print(Torque*1.356)
if year < 2050:
      Torque = Torque * ex_0d996p_year50R

Torque = Torque * SubComponent_Valve_PowerRating*1.356
print(Torque)

278.6888382950357
219.99891665758014


In [ ]:
bore = 50.00
stroke = 150-bore
max_hp = 0
best_bore = 0
best_stroke = 0
while bore <= 100:
  cal_cc,cal_torque,cal_hp,cal_cost = cal_optimal_performance(bore,stroke)
  if hp > max_hp:
    best_bore = bore
    best_stroke = stroke
    max_hp = hp
  bore += 0.01
  stroke = 150-bore

TypeError: '>' not supported between instances of 'tuple' and 'int'

In [ ]:
print(best_bore,best_stroke,max_hp)